# 10 · Advanced RAG: Chunking Strategies

Fixed-size, recursive, sentence, semantic, parent-document and late chunking compared.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/10_advanced_rag_chunking.ipynb)

**Covered in this notebook**

- Advanced RAG Agents
- 2-Recursive Chunking
- 3-Sentence Based Chunking
- 4-Semantic Chunking
- 5-Parent Document Chunking:

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

# Advanced RAG Agents

## Chunking

#1-Fixed Size Chunking

In [ ]:
!pip install langchain-text-splitters -q

In [ ]:
#Fixed Size Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
Photosynthesis is the process by which plants convert sunlight into food.
It occurs in the chloroplasts. The plant absorbs carbon dioxide and water.
Using light energy, it produces glucose and oxygen. This process sustains most life on Earth.
Glucose is used by the plant for energy and growth. Oxygen is released as a byproduct.
"""

# Create Recursive Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,       # Maximum size of each chunk
    chunk_overlap=20,     # Characters repeated in the next chunk
    length_function=len   # Counts characters
)

# Split the text into chunks
chunks = text_splitter.create_documents([text])

# Print each chunk
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} (Size: {len(chunk.page_content)}) ---")
    print(chunk.page_content)
    print()

--- Chunk 1 (Size: 73) ---
Photosynthesis is the process by which plants convert sunlight into food.

--- Chunk 2 (Size: 74) ---
It occurs in the chloroplasts. The plant absorbs carbon dioxide and water.

--- Chunk 3 (Size: 93) ---
Using light energy, it produces glucose and oxygen. This process sustains most life on Earth.

--- Chunk 4 (Size: 86) ---
Glucose is used by the plant for energy and growth. Oxygen is released as a byproduct.



# 2-Recursive Chunking

In [ ]:
# Recursive Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
Photosynthesis is the process by which plants convert sunlight into food.
It occurs in the chloroplasts. The plant absorbs carbon dioxide and water.
Using light energy, it produces glucose and oxygen. This process sustains most life on Earth.
Glucose is used by the plant for energy and growth. Oxygen is released as a byproduct.
"""

# Initialize the recursive text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    # The hierarchy of separators it tries to use (this is the default)
    separators=["\n\n", "\n", ".", " ", ""]
)

# Create the chunks
chunks = text_splitter.create_documents([text])

# Print the results
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} (Size: {len(chunk.page_content)}) ---")
    print(chunk.page_content)
    print()

--- Chunk 1 (Size: 73) ---
Photosynthesis is the process by which plants convert sunlight into food.

--- Chunk 2 (Size: 74) ---
It occurs in the chloroplasts. The plant absorbs carbon dioxide and water.

--- Chunk 3 (Size: 93) ---
Using light energy, it produces glucose and oxygen. This process sustains most life on Earth.

--- Chunk 4 (Size: 86) ---
Glucose is used by the plant for energy and growth. Oxygen is released as a byproduct.



# 3-Sentence Based Chunking

In [ ]:
!pip install nltk -q

In [ ]:
# Sentence Based Chunking
import nltk

nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize

text = """
Dr. Smith published findings on photosynthesis in 2023. The process, i.e. converting light to food,
occurs in chloroplasts. Plants absorb CO2 and H2O. They release O2 as a byproduct.
This sustains most life on Earth. Glucose powers plant growth and reproduction.
"""

# Split text into sentences
sentences = sent_tokenize(text)

# Group 2 sentences per chunk
chunk_size = 2

chunks = []

for i in range(0, len(sentences), chunk_size):
    chunk = " ".join(sentences[i:i + chunk_size])
    chunks.append(chunk)

# Print chunks
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}:")
    print(chunk)
    print()

Chunk 1:

Dr. Smith published findings on photosynthesis in 2023. The process, i.e.

Chunk 2:
converting light to food,
occurs in chloroplasts. Plants absorb CO2 and H2O.

Chunk 3:
They release O2 as a byproduct. This sustains most life on Earth.

Chunk 4:
Glucose powers plant growth and reproduction.



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# 4-Semantic Chunking

In [ ]:
!pip install langchain-experimental langchain-huggingface sentence-transformers -q

In [ ]:
# Semantic Chunking -> Break where meaning changes
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

text = """
Photosynthesis is the process plants use to convert sunlight into glucose.
It requires carbon dioxide, water, and light energy.
Chlorophyll in the chloroplasts absorbs the light.

The French Revolution began in 1789 and fundamentally changed European society.
Louis XVI was executed in 1793. The revolution introduced ideals of liberty and equality.
Napoleon rose to power in its aftermath and reshaped the continent.

Machine learning is a subset of artificial intelligence.
Neural networks learn patterns from data through gradient descent.
Transformers have become the dominant architecture since 2017.
"""

# Hugging Face embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Semantic chunker
splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70
)

# Create chunks
chunks = splitter.split_text(text)

# Print chunks
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk)

print("\nTotal chunks:", len(chunks))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- Chunk 1 ---

Photosynthesis is the process plants use to convert sunlight into glucose. It requires carbon dioxide, water, and light energy.

--- Chunk 2 ---
Chlorophyll in the chloroplasts absorbs the light. The French Revolution began in 1789 and fundamentally changed European society. Louis XVI was executed in 1793. The revolution introduced ideals of liberty and equality.

--- Chunk 3 ---
Napoleon rose to power in its aftermath and reshaped the continent. Machine learning is a subset of artificial intelligence. Neural networks learn patterns from data through gradient descent. Transformers have become the dominant architecture since 2017.

--- Chunk 4 ---


Total chunks: 4


# 5-Parent Document Chunking:

In [ ]:
# Parent Document Chunking:

In [ ]:
!pip install langchain-classic langchain-core -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.3 MB/s eta 0:00:00


In [ ]:
!pip install langchain langchain-community langchain-chroma langchain-huggingface sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7

In [ ]:
# UPDATED IMPORTS
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 1. Create a sample document
text = """
Photosynthesis is the process by which plants convert sunlight into food. It occurs in the chloroplasts.
The plant absorbs carbon dioxide and water. Using light energy, it produces glucose and oxygen.
The Industrial Revolution transitioned human society from agrarian to urban.
Factories replaced hand tools, and steam engines powered new machinery.
This period saw massive economic and social shifts across the world.
"""
docs = [Document(page_content=text)]

# 2. Set up the two splitters
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

# 3. Set up Storage (Using the modern import for InMemoryStore)
vectorstore = Chroma(
    collection_name="split_parents",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)
docstore = InMemoryStore()

# 4. Initialize the Parent Document Retriever (Using the classic import)
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 5. Add documents
retriever.add_documents(docs)

# 6. Test the retrieval
query = "Where does photosynthesis occur?"
retrieved_docs = retriever.invoke(query)

print(f"Found {len(retrieved_docs)} parent document(s).\n")
print("--- Context Provided to LLM ---")
print(retrieved_docs[0].page_content)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Found 1 parent document(s).

--- Context Provided to LLM ---
Photosynthesis is the process by which plants convert sunlight into food. It occurs in the chloroplasts.
The plant absorbs carbon dioxide and water. Using light energy, it produces glucose and oxygen.
The Industrial Revolution transitioned human society from agrarian to urban.
Factories replaced hand tools, and steam engines powered new machinery.
This period saw massive economic and social shifts across the world.


#6-Sentence Window Retrieval

In [ ]:
# Install required libraries
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer, util
import re


# Sample document
text = """
Photosynthesis is the process by which plants convert sunlight into food.
It occurs mainly in the chloroplasts of plant cells.
Plants absorb carbon dioxide and water during this process.
Using light energy, plants produce glucose and oxygen.
Glucose is used by the plant for energy and growth.
Oxygen is released into the atmosphere as a byproduct.
"""


# Split document into individual sentences
sentences = re.split(r'(?<=[.!?])\s+', text.strip())

print("Total sentences:", len(sentences))


# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")


# Create embeddings for all sentences
sentence_embeddings = model.encode(
    sentences,
    convert_to_tensor=True
)


# User query
query = "What does photosynthesis produce?"


# Convert query into embedding
query_embedding = model.encode(
    query,
    convert_to_tensor=True
)


# Find the most relevant sentence
scores = util.cos_sim(query_embedding, sentence_embeddings)[0]
best_index = scores.argmax().item()


# Define how many surrounding sentences we want
window_size = 1


# Get previous + relevant + next sentence
start = max(0, best_index - window_size)
end = min(len(sentences), best_index + window_size + 1)

window = sentences[start:end]


# Display the retrieved context
print("\nRelevant Sentence:")
print(sentences[best_index])

print("\nSentence Window:")
for sentence in window:
    print(sentence)

Total sentences: 6


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Relevant Sentence:
Photosynthesis is the process by which plants convert sunlight into food.

Sentence Window:
Photosynthesis is the process by which plants convert sunlight into food.
It occurs mainly in the chloroplasts of plant cells.


#7-Late Chunking using BERT

In [ ]:
# Late Chunking using BERT

!pip install -q -U "transformers>=4.41,<4.46" "tokenizers>=0.19,<0.21" sentencepiece accelerate
print("\nLate Chunking completed successfully! 🚀")


Late Chunking completed successfully! 🚀


In [ ]:
import transformers
import tokenizers

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Transformers: 4.45.2
Tokenizers: 0.20.3


In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = "jinaai/jina-embeddings-v2-base-en"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True
)

print("Jina model loaded successfully!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

modeling_bert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-bert-implementation:
- modeling_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/275M [00:00<?, ?B/s]

Jina model loaded successfully!


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import torch

text = """
Photosynthesis is the process by which plants convert sunlight into food.
It occurs in the chloroplasts. The plant absorbs carbon dioxide and water.
Using light energy, it produces glucose and oxygen.
This process sustains most life on Earth.

The Industrial Revolution transitioned human society from agrarian to urban.
Factories replaced hand tools, and steam engines powered new machinery.
This period saw massive economic and social shifts across the world.
"""

docs = [Document(page_content=text)]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    add_start_index=True
)

chunks = splitter.split_documents(docs)

print("Number of chunks:", len(chunks))

Number of chunks: 4


In [ ]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    return_offsets_mapping=True,
    truncation=True,
    max_length=8192
)

offsets = inputs.pop("offset_mapping")[0]

with torch.no_grad():
    outputs = model(**inputs)

token_embeddings = outputs.last_hidden_state[0]

print("Token embeddings shape:", token_embeddings.shape)

Token embeddings shape: torch.Size([89, 768])


In [ ]:
chunk_embeddings = []

for chunk in chunks:

    start = chunk.metadata["start_index"]
    end = start + len(chunk.page_content)

    token_ids = [
        i for i, (s, e) in enumerate(offsets.tolist())
        if e > start and s < end
    ]

    chunk_embedding = token_embeddings[token_ids].mean(dim=0)

    chunk_embeddings.append(chunk_embedding)

In [ ]:
print("\n" + "=" * 50)
print("LATE CHUNKING USING JINA EMBEDDINGS V2")
print("=" * 50)

print("Number of chunks:", len(chunk_embeddings))
print("Embedding size:", chunk_embeddings[0].shape[0])

for i, (chunk, embedding) in enumerate(
    zip(chunks, chunk_embeddings)
):

    print(f"\nChunk {i + 1}:")
    print(chunk.page_content.strip())
    print("Embedding shape:", embedding.shape)

print("\nLate Chunking completed successfully! 🚀")


LATE CHUNKING USING JINA EMBEDDINGS V2
Number of chunks: 4
Embedding size: 768

Chunk 1:
Photosynthesis is the process by which plants convert sunlight into food.
It occurs in the chloroplasts. The plant absorbs carbon dioxide and water.
Embedding shape: torch.Size([768])

Chunk 2:
Using light energy, it produces glucose and oxygen.
This process sustains most life on Earth.
Embedding shape: torch.Size([768])

Chunk 3:
The Industrial Revolution transitioned human society from agrarian to urban.
Factories replaced hand tools, and steam engines powered new machinery.
Embedding shape: torch.Size([768])

Chunk 4:
This period saw massive economic and social shifts across the world.
Embedding shape: torch.Size([768])

Late Chunking completed successfully! 🚀
